In [25]:
import requests, pdfplumber, json, time
OLLAMA_URL = "http://localhost:11434"
GEN_MODEL  = "llama3:latest"

# check what models are installed right now
try:
    tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
    available = [m['name'] for m in tags.get('models', [])]
    print("Available models:", available)
    if not any(GEN_MODEL in m for m in available):
        print(f"[!] '{GEN_MODEL}' not found. Run:  ollama pull {GEN_MODEL}")
    else:
        print(f"OK  '{GEN_MODEL}' is ready")
except Exception as e:
    print(f"[!] Cannot reach ollama: {e}  — run: ollama serve")

Available models: ['llama3:latest']
OK  'llama3:latest' is ready


In [26]:
PDF_PATH = "C:\\Users\\Mateti sai pranay\\Downloads\\.net\\python\\re_vamp\\pdf-text_pipline\\docs_images\\invoices\\medical_similar\\shiva\\invoice1.pdf"

text = ""
tables = []
with pdfplumber.open(PDF_PATH) as pdf:
    for page in pdf.pages:
        text += page.extract_text() or ""
        for t in page.extract_tables():
            if t:
                tables.append({"page": page.page_number, "rows": t})

def clean_table(rows):
    out = []
    for row in rows:
        cells = [str(c).strip() for c in row if c is not None and str(c).strip()]
        if cells:
            out.append(" | ".join(cells))
    return "\n".join(out)

table_text = clean_table(tables[0]["rows"]) if tables else ""
print(f"text={len(text)} chars   table={len(table_text)} chars")

text=2542 chars   table=2621 chars


## LLM helper

In [27]:
extracted = {}

def ask(label, schema_json, context):
    prompt = (
        f"Extract {label} from this GST invoice.\n"
        "Return ONLY valid JSON matching the schema. Numbers as numbers, null for missing.\n\n"
        f"INVOICE DATA:\n{context[:1200]}\n\n"
        f"SCHEMA:\n{schema_json}"
    )
    payload = {
        "model": GEN_MODEL,
        "prompt": prompt,
        "stream": True,
        "format": "json",
        "options": {"temperature": 0.0, "num_ctx": 1536}
    }
    try:
        r = requests.post(
            f"{OLLAMA_URL}/api/generate",
            json=payload, stream=True, timeout=(10, 300)
        )
        # 404 = model not pulled yet
        if r.status_code == 404:
            tags = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()
            available = [m['name'] for m in tags.get('models', [])]
            print(f"[!] 404 — model '{GEN_MODEL}' not found on ollama.")
            print(f"    Pull it:   ollama pull {GEN_MODEL}")
            print(f"    Installed: {available}")
            return {}
        r.raise_for_status()
        buf = []
        for line in r.iter_lines():
            if line:
                chunk = json.loads(line)
                buf.append(chunk.get("response", ""))
                if chunk.get("done"):
                    break
        raw = "".join(buf).strip()
        if raw.startswith("```"):
            raw = raw.split("\n", 1)[-1].rsplit("```", 1)[0].strip()
        result = json.loads(raw)
        extracted.update(result)
        print(f"  OK  {label}")
        return result
    except requests.exceptions.ReadTimeout:
        print(f"  [!] TIMEOUT — model too slow.  Fix: ollama pull qwen2.5:3b")
    except requests.exceptions.ConnectionError:
        print(f"  [!] CONNECTION ERROR — run: ollama serve")
    except requests.exceptions.HTTPError as e:
        print(f"  [!] HTTP {e.response.status_code} error: {e}")
    except json.JSONDecodeError as e:
        print(f"  [!] JSON parse failed for {label}: {e}")
    return {}

print("ask() ready")

ask() ready


## Extract 1 — Seller Info

In [28]:
t0 = time.time()
ask(
    label="seller info",
    schema_json='{"seller_info":{"shop_name":"","address":"","phone":"","dl_no":"","gstin":""}}',
    context=text
)
print(f"  {time.time()-t0:.1f}s")
print(extracted.get("seller_info"))

  OK  seller info
  44.2s
{'shop_name': 'SRI SHIVA SAI POULTRY & VETERINARY NEEDS', 'address': 'STALL NO 14 BUS STAND COMPLEX KOTHAGUDEM, D.NO.7-146/1, ROOM NO-1,KOTHAPETA ROAD, BHADRADRI KOTHAGUDEM DIST - 507101.', 'phone': ['9848091819', '9866514042'], 'dl_no': 'TG/22/02/2001-20590,20591,20592,593', 'gstin': '36AAOFS3456N1ZG'}


## Extract 2 — Invoice Info

In [29]:
t0 = time.time()
ask(
    label="invoice info",
    schema_json='{"invoice_info":{"invoice_no":"","invoice_date":"","due_date":""}}',
    context=text
)
print(f"  {time.time()-t0:.1f}s")
print(extracted.get("invoice_info"))

  OK  invoice info
  106.6s
{'invoice_no': 'B001000786', 'invoice_date': '13-07-2023', 'due_date': '03-08-2023'}


## Extract 3 — Buyer Info

In [30]:
t0 = time.time()
ask(
    label="buyer info",
    schema_json='{"buyer_info":{"party_name":"","address":"","phone":"","state_code":"","dl_no":""}}',
    context=text
)
print(f"  {time.time()-t0:.1f}s")
print(extracted.get("buyer_info"))

  OK  buyer info
  146.6s
{'party_name': 'SRI SHIVA SAI POULTRY & VETERINARY NEEDS', 'address': 'STALL NO 14 BUS STAND COMPLEX KOTHAGUDEM\nD.NO.7-146/1, ROOM NO-1,KOTHAPETA ROAD,\nBHADRADRI KOTHAGUDEM DIST - 507101.\nCREDIT DAMMAPETA(V),DAMMAPETA(M),BHADRADRI(D),TG,IND', 'phone': '9848091819\nPHONE. : 9866514042', 'state_code': '36AAOFS3456N1ZG', 'dl_no': 'TG/22/02/2001-20590,20591,20592,593'}


## Extract 4 — Products

In [31]:
t0 = time.time()
ask(
    label="product names",
    schema_json='{"products":["product name string"]}',
    context=table_text
)
print(f"  {time.time()-t0:.1f}s")
for i, p in enumerate(extracted.get("products", []), 1):
    print(f"  {i:2}. {p}")

  OK  product names
  132.9s
   1. {'Product Name': 'ZENCLONEX-I ORAL 1LIIT'}
   2. {'Product Name': 'MORAL INJ'}
   3. {'Product Name': 'ULTROX LIQ 100ML'}
   4. {'Product Name': 'CHRCOL SUSP'}
   5. {'Product Name': 'GOLD BRIO-R SYP 500ML'}
   6. {'Product Name': 'MEDIWON H ORAL 100ML'}
   7. {'Product Name': 'ATROVIN INJ 100ML'}
   8. {'Product Name': 'DEXOVET INJ 30'}
   9. {'Product Name': 'TOXOL 1LITER'}


## Extract 5 — Amounts

In [ ]:
t0 = time.time()
ask(
    label="net amounts per product",
    schema_json='{"amounts":[0.0]}',
    context=table_text
)
print(f"  {time.time()-t0:.1f}s")
for i, a in enumerate(extracted.get("amounts", []), 1):
    print(f"  {i:2}. {a}")

## Extract 6 — Totals

In [ ]:
t0 = time.time()
ask(
    label="invoice totals",
    schema_json='{"totals":{"Grand Total":0,"SGST":0,"CGST":0}}',
    context=table_text
)
print(f"  {time.time()-t0:.1f}s")
print(extracted.get("totals"))

  OK  invoice totals
  104.3s
{'grand_total': 23.0, 'total_sgst': 0.0, 'total_cgst': 0.0}


## Extract 7 — Bank Details

In [ ]:
t0 = time.time()
ask(
    label="bank details",
    schema_json='{"bank_details":{"bank_name":"","account_no":"","ifsc":"","branch":""}}',
    context=table_text
)
print(f"  {time.time()-t0:.1f}s")
print(extracted.get("bank_details"))

## Final Summary

In [ ]:
def section(title):
    print(f"\n{'='*52}\n  {title}\n{'='*52}")

section("LIST 1 -- SELLER INFO")
for k, v in extracted.get("seller_info", {}).items():
    print(f"  {k:15s}: {v}")

section("LIST 2 -- INVOICE INFO")
for k, v in extracted.get("invoice_info", {}).items():
    print(f"  {k:15s}: {v}")

section("LIST 3 -- BUYER INFO")
for k, v in extracted.get("buyer_info", {}).items():
    print(f"  {k:15s}: {v}")

section("LIST 4 -- PRODUCTS")
for i, p in enumerate(extracted.get("products", []), 1):
    print(f"  {i:2}. {p}")

section("LIST 5 -- AMOUNTS")
for i, a in enumerate(extracted.get("amounts", []), 1):
    print(f"  {i:2}. {a}")

section("LIST 6 -- TOTALS")
for k, v in extracted.get("totals", {}).items():
    print(f"  {k:15s}: {v}")

section("LIST 7 -- BANK DETAILS")
for k, v in extracted.get("bank_details", {}).items():
    print(f"  {k:15s}: {v}")